# 1T Feature-Family Classifier Windows to Options

This notebook compares three ways of feeding equity classifier signals into the ThetaData option selector:

1. `ensemble_mean`: the averaged classifier signal.
2. `top_feature_families`: the best individual feature-family signals ranked by equity shared-book Sharpe.
3. `all_feature_families`: every individual feature-family signal.

The goal is to test whether the options pipeline was starved because the ensemble creates too few trade windows, or because the current equity classifier windows are not good option entries.

Daily non-events are not included. Each row is an actual classifier entry/exit window.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display, Markdown

REPO_ROOT = Path.cwd()
if REPO_ROOT.name != 'quant-orchestrator':
    REPO_ROOT = next(parent for parent in Path.cwd().resolve().parents if parent.name == 'quant-orchestrator')
WAREHOUSE_ROOT = REPO_ROOT.parent / 'quant-warehouse'
for path in (REPO_ROOT, WAREHOUSE_ROOT):
    resolved = str(path.resolve())
    if resolved in sys.path:
        sys.path.remove(resolved)
    sys.path.insert(0, resolved)
for module_name in list(sys.modules):
    if module_name == 'quant_orchestrator' or module_name.startswith('quant_orchestrator.'):
        del sys.modules[module_name]
    if module_name == 'quant_warehouse' or module_name.startswith('quant_warehouse.'):
        del sys.modules[module_name]

from quant_orchestrator.research_tools import (
    OptopsyExecutionConfig,
    OptionMvBasketConfig,
    OptionRetrievalConfig,
    OracleOptionExperimentConfig,
    SharedSplitConfig,
    build_classifier_signal_trade_windows,
    rank_option_window_strategy_sources,
    run_trade_window_option_experiment,
)

pd.set_option('display.max_columns', 160)
pd.set_option('display.width', 220)
print('repo_root', REPO_ROOT)
print('warehouse_root', WAREHOUSE_ROOT)

repo_root /home/jlee153232/PycharmProjects/quant-orchestrator
warehouse_root /home/jlee153232/PycharmProjects/quant-warehouse


In [2]:
SCORE_RUN = REPO_ROOT / 'artifacts/orchestrator/files/ml_trading_gpu_rf_shared_book_1t_dagster_smoke/run_f6b98a7b10ab4c7cb26ef41ece402750'
SCORE_PATH = SCORE_RUN / 'strategy_scores.csv'
BACKTEST_SUMMARY_PATH = SCORE_RUN / 'backtest_summary.csv'

PRICE_START = '2018-01-01'
PRICE_END = '2026-06-24'
ENTRY_THRESHOLD = 0.50
EXIT_THRESHOLD = 0.50
TOP_K = 5
TOP_FAMILY_COUNT = 8
RUN_EXPERIMENTS_IF_MISSING = True

scores = pd.read_csv(SCORE_PATH)
backtest_summary = pd.read_csv(BACKTEST_SUMMARY_PATH)
ranked_sources = rank_option_window_strategy_sources(
    backtest_summary,
    variant='long_short',
    top_k=TOP_K,
    framework='zipline_shared_book_native',
    min_signal_events=1,
)
display(ranked_sources[['strategy_source', 'source', 'family', 'total_return', 'sharpe', 'max_drawdown', 'signal_events']].head(20))
print('score_rows', len(scores), 'symbols', scores['symbol'].nunique(), 'strategy_sources', scores['strategy_source'].nunique())

,strategy_source,source,family,total_return,sharpe,max_drawdown,signal_events
0,financetoolkit.ft_growth_balance,financetoolkit,ft_growth_balance,2.6699,1.3061,-0.1901,7
1,financetoolkit.ft_ratios_efficiency,financetoolkit,ft_ratios_efficiency,3.4723,1.2777,-0.2523,9
2,financetoolkit.ft_growth_cash,financetoolkit,ft_growth_cash,2.7605,1.1959,-0.2592,9
3,financetoolkit.ft_ratios_liquidity,financetoolkit,ft_ratios_liquidity,2.2346,0.9671,-0.2854,11
4,financetoolkit.ft_ratios_solvency,financetoolkit,ft_ratios_solvency,1.7217,0.9627,-0.2523,7
5,financetoolkit.ft_ratios_valuation,financetoolkit,ft_ratios_valuation,1.7217,0.9627,-0.2523,7
6,financetoolkit.ft_ratios_profitability,financetoolkit,ft_ratios_profitability,1.4672,0.8229,-0.2522,7
7,financetoolkit.ft_growth_income,financetoolkit,ft_growth_income,1.4311,0.8160,-0.2522,5
8,fmp.fmp_daily_ev_yield,fmp,fmp_daily_ev_yield,-0.0976,0.0155,-0.5174,339
9,fmp.fmp_cash_mcap,fmp,fmp_cash_mcap,-0.3396,-0.2250,-0.6090,45


score_rows 338416 symbols 13 strategy_sources 16


## Build Trade Windows

In [3]:
all_family_sources = tuple(sorted(src for src in scores['strategy_source'].dropna().unique() if src != 'ensemble_mean'))
top_family_sources = tuple(ranked_sources['strategy_source'].head(TOP_FAMILY_COUNT).tolist())
source_groups = {
    'ensemble_mean': ('ensemble_mean',),
    'top_feature_families': top_family_sources,
    'all_feature_families': all_family_sources,
}

window_frames = {}
window_summary_rows = []
for name, sources_for_group in source_groups.items():
    windows = build_classifier_signal_trade_windows(
        scores,
        strategy_sources=sources_for_group,
        variant='long_short',
        top_k=TOP_K,
        entry_threshold=ENTRY_THRESHOLD,
        exit_threshold=EXIT_THRESHOLD,
    )
    window_frames[name] = windows
    window_summary_rows.append({
        'group': name,
        'sources': len(sources_for_group),
        'windows': len(windows),
        'symbols': windows['symbol'].nunique() if not windows.empty else 0,
        'min_entry_date': windows['entry_date'].min() if not windows.empty else pd.NaT,
        'max_entry_date': windows['entry_date'].max() if not windows.empty else pd.NaT,
    })
window_summary = pd.DataFrame(window_summary_rows)
display(window_summary)
for name, windows in window_frames.items():
    print()
    print(name)
    if windows.empty:
        print('no windows')
    else:
        display(windows.groupby('strategy_source').size().sort_values(ascending=False).head(20).rename('windows').reset_index())

,group,sources,windows,symbols,min_entry_date,max_entry_date
0,ensemble_mean,1,11,10,2020-01-02,2026-03-27
1,top_feature_families,8,51,11,2020-01-02,2026-02-10
2,all_feature_families,15,592,13,2020-01-02,2026-06-15



ensemble_mean


,strategy_source,windows
0,ensemble_mean,11



top_feature_families


,strategy_source,windows
0,financetoolkit.ft_ratios_liquidity,8
1,financetoolkit.ft_growth_cash,7
2,financetoolkit.ft_ratios_efficiency,7
3,financetoolkit.ft_growth_balance,6
4,financetoolkit.ft_ratios_profitability,6
5,financetoolkit.ft_ratios_solvency,6
6,financetoolkit.ft_ratios_valuation,6
7,financetoolkit.ft_growth_income,5



all_feature_families


,strategy_source,windows
0,fmp.fmp_daily_ev_multiple,195
1,fmp.fmp_daily_ev_yield,172
2,fmp.fmp_daily_mcap_yield,51
3,fmp.fmp_daily_mcap_multiple,42
4,fmp.fmp_income_mcap,33
5,fmp.fmp_cash_mcap,25
6,fmp.fmp_balance_mcap,23
7,financetoolkit.ft_ratios_liquidity,8
8,financetoolkit.ft_growth_cash,7
9,financetoolkit.ft_ratios_efficiency,7


## Run Or Load Option Experiments

In [4]:
def option_config(name: str) -> OracleOptionExperimentConfig:
    return OracleOptionExperimentConfig(
        experiment_name=f'classifier_1t_{name}_option_windows',
        symbols=tuple(sorted(scores['symbol'].dropna().astype(str).str.upper().unique())),
        price_start=PRICE_START,
        price_end=PRICE_END,
        split=SharedSplitConfig(insample_end='2024-12-31'),
        retrieval=OptionRetrievalConfig(
            option_universe='full_chain_actions',
            target_dte=90,
            require_affordable=False,
            min_entry_mid=0.0,
            max_entry_spread_pct=10_000.0,
            max_abs_moneyness=10_000.0,
            max_candidates_per_trade=1_000_000,
        ),
        execution=OptopsyExecutionConfig(capital=100_000.0, quantity=1, max_positions=5, multiplier=100, selector='first'),
        mv_basket=OptionMvBasketConfig(enabled=True, max_legs=4, min_predicted_weight=0.02),
        artifact_dir=str(REPO_ROOT / f'artifacts/options/classifier_1t_{name}_option_windows/latest'),
        log_mlflow=False,
    )

def load_outputs(name: str):
    base = REPO_ROOT / f'artifacts/options/classifier_1t_{name}_option_windows/latest'
    if not (base / 'selector_summary.csv').exists():
        return None
    return {
        'selector_summary': pd.read_csv(base / 'selector_summary.csv'),
        'optopsy_summary': pd.read_csv(base / 'optopsy_summary.csv'),
        'oracle_trades': pd.read_parquet(base / 'oracle_trades.parquet'),
        'option_panel': pd.read_parquet(base / 'option_candidate_panel.parquet'),
        'train_panel': pd.read_parquet(base / 'train_panel.parquet'),
        'eval_panel': pd.read_parquet(base / 'eval_panel.parquet'),
    }

outputs = {}
# Map prior artifact names into the comparison names used by this notebook.
artifact_aliases = {
    'ensemble_mean': REPO_ROOT / 'artifacts/options/classifier_1t_option_signal_windows/latest',
    'top_feature_families': REPO_ROOT / 'artifacts/options/classifier_1t_top_feature_family_option_windows/latest',
    'all_feature_families': REPO_ROOT / 'artifacts/options/classifier_1t_feature_family_option_windows/latest',
}
for name, windows in window_frames.items():
    alias = artifact_aliases[name]
    canonical = REPO_ROOT / f'artifacts/options/classifier_1t_{name}_option_windows/latest'
    if alias.exists() and not canonical.exists():
        canonical.parent.mkdir(parents=True, exist_ok=True)
        # Keep notebook side effects small: read from alias rather than copy artifacts.
    loaded = None
    base = alias if alias.exists() else canonical
    if (base / 'selector_summary.csv').exists():
        loaded = {
            'selector_summary': pd.read_csv(base / 'selector_summary.csv'),
            'optopsy_summary': pd.read_csv(base / 'optopsy_summary.csv'),
            'oracle_trades': pd.read_parquet(base / 'oracle_trades.parquet'),
            'option_panel': pd.read_parquet(base / 'option_candidate_panel.parquet'),
            'train_panel': pd.read_parquet(base / 'train_panel.parquet'),
            'eval_panel': pd.read_parquet(base / 'eval_panel.parquet'),
        }
    elif RUN_EXPERIMENTS_IF_MISSING:
        result = run_trade_window_option_experiment(option_config(name), windows)
        loaded = {
            'selector_summary': result.selector_summary,
            'optopsy_summary': result.optopsy_summary,
            'oracle_trades': result.oracle_trades,
            'option_panel': result.option_panel,
            'train_panel': result.train_panel,
            'eval_panel': result.eval_panel,
        }
    outputs[name] = loaded
    print(name, 'loaded' if loaded is not None else 'missing')

ensemble_mean loaded
top_feature_families loaded
all_feature_families loaded


## Compare Results

In [5]:
comparison_rows = []
selector_tables = []
for name, data in outputs.items():
    if data is None:
        continue
    selector = data['selector_summary'].copy()
    optopsy = data['optopsy_summary'].copy()
    selector['group'] = name
    optopsy['group'] = name
    selector_tables.append(selector)
    for _, row in optopsy.iterrows():
        comparison_rows.append({
            'group': name,
            'selector': row.get('selector'),
            'closed_trades': row.get('closed_trades'),
            'final_equity': row.get('final_equity'),
            'total_return': row.get('total_return'),
            'max_drawdown': row.get('max_drawdown'),
            'sharpe_ratio': row.get('sharpe_ratio'),
        })
comparison = pd.DataFrame(comparison_rows)
selector_detail = pd.concat(selector_tables, ignore_index=True) if selector_tables else pd.DataFrame()
display(comparison.sort_values(['group', 'total_return'], ascending=[True, False]))
display(selector_detail[['group', 'selector', 'trades', 'mean_return', 'median_return', 'win_rate']])

,group,selector,closed_trades,final_equity,total_return,max_drawdown,sharpe_ratio
16,all_feature_families,fixed_near_atm,48,98659.500000,-0.013405,-0.114963,-0.022356
18,all_feature_families,lowest_spread,45,96247.000000,-0.037530,-0.165081,-0.124051
20,all_feature_families,oracle_mv_basket,51,94792.432133,-0.052076,-0.158827,-0.230656
19,all_feature_families,model_mv_basket,51,91426.090193,-0.085739,-0.134676,-0.315121
15,all_feature_families,oracle_best_possible,50,89473.000000,-0.105270,-0.180474,-0.380722
17,all_feature_families,highest_liquidity,36,83443.000000,-0.165570,-0.200266,-0.970415
14,all_feature_families,model_ranker,57,80577.500000,-0.194225,-0.230485,-0.792298
6,ensemble_mean,oracle_mv_basket,3,110317.938848,0.103179,0.000000,1.129574
0,ensemble_mean,model_ranker,3,108829.500000,0.088295,0.000000,1.249052
1,ensemble_mean,oracle_best_possible,3,108829.500000,0.088295,0.000000,1.249052


,group,selector,trades,mean_return,median_return,win_rate
0,ensemble_mean,model_ranker,3,2.149260,1.657005,1.000000
1,ensemble_mean,oracle_best_possible,3,2.149260,1.657005,1.000000
2,ensemble_mean,fixed_near_atm,3,0.172426,0.158879,0.666667
3,ensemble_mean,highest_liquidity,3,0.085333,-0.101466,0.333333
4,ensemble_mean,lowest_spread,3,0.237648,0.355482,0.666667
5,ensemble_mean,model_mv_basket,0,NaN,NaN,NaN
6,ensemble_mean,oracle_mv_basket,3,1.450640,1.370158,1.000000
7,top_feature_families,model_ranker,2,-1.000000,-1.000000,0.000000
8,top_feature_families,oracle_best_possible,2,0.231932,0.231932,0.500000
9,top_feature_families,fixed_near_atm,2,-1.000000,-1.000000,0.000000


## Written Analysis

In [6]:
lines = [
    '## Analysis',
    '',
    f'- Ensemble mean generated {int(window_summary.loc[window_summary["group"].eq("ensemble_mean"), "windows"].iloc[0]):,} classifier windows before option coverage filtering.',
    f'- Top {TOP_FAMILY_COUNT} feature families generated {int(window_summary.loc[window_summary["group"].eq("top_feature_families"), "windows"].iloc[0]):,} windows.',
    f'- All individual feature families generated {int(window_summary.loc[window_summary["group"].eq("all_feature_families"), "windows"].iloc[0]):,} windows.',
    '',
    'The ensemble is too smooth for options: it holds positions for long periods and produces too few entry/exit events.',
    'The top equity feature-family subset is directionally higher quality, but after the 2024 split it leaves too few option eval trades to train or judge the option selector.',
    'All feature families solve the data starvation problem, but they introduce lower-quality equity windows. In the current 1T run, the all-family option selectors lost money after Optopsy execution, including the learned single-leg selector and learned MV basket selector.',
    '',
    'This means the next fix should be signal-window construction, not a more complex option model. The option selector needs enough windows, but those windows still need to be economically good equity entries.',
    '',
    'Recommended next experiment: generate option-specific equity windows from individual feature families with shorter planned holding horizons and option-liquidity-aware filters, then rerun the same option selector/MV basket path.',
]
analysis = chr(10).join(lines)
Markdown(analysis)

## Analysis

- Ensemble mean generated 11 classifier windows before option coverage filtering.
- Top 8 feature families generated 51 windows.
- All individual feature families generated 592 windows.

The ensemble is too smooth for options: it holds positions for long periods and produces too few entry/exit events.
The top equity feature-family subset is directionally higher quality, but after the 2024 split it leaves too few option eval trades to train or judge the option selector.
All feature families solve the data starvation problem, but they introduce lower-quality equity windows. In the current 1T run, the all-family option selectors lost money after Optopsy execution, including the learned single-leg selector and learned MV basket selector.

This means the next fix should be signal-window construction, not a more complex option model. The option selector needs enough windows, but those windows still need to be economically good equity entries.

Recommended next experiment: generate option-specific equity windows from individual feature families with shorter planned holding horizons and option-liquidity-aware filters, then rerun the same option selector/MV basket path.